# Авито тестовое


**BM25 top-8000 → CatBoostRanker → top-50 → `answer.csv`**

Модель CatBoost обучается на 2000 train-контекстах. Для каждого обучающего запроса BM25 берёт top-500, после чего сохраняются все положительные примеры, 20 самых сложных отрицательных и до 30 случайных отрицательных. На инференсе пул расширяется до 8000 кандидатов, а CatBoost выбирает итоговые 50.

Контрольные результаты:
- локальная validation: **Recall@50 ≈ 0.754395** для BM25 top-8000 → CatBoost top-50;
- скрытый benchmark: **Recall@50 = 0.745558**.

## Входные файлы
в `/content`:
- `train.parquet`
- `benchmark_queries.parquet`
- `benchmark_items.parquet`

GPU не требуется: BM25 и CatBoost здесь работают на CPU.

Используются open-source библиотеки `bm25s`, `PyStemmer`, `CatBoost`, `pandas`, `NumPy`, `PyArrow`.


In [ ]:
%pip -q install bm25s==0.2.14 PyStemmer==3.0.0 catboost==1.2.8 pandas pyarrow numpy


## 1. Конфигурация и загрузка данных
Для локальной проверки берутся 1000 validation-контекстов; CatBoost обучается на 2000 train-контекстах.


In [ ]:
from pathlib import Path
import gc
import json
import re

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import bm25s
import Stemmer
from catboost import CatBoostRanker, Pool

DATA = Path('/content')
OUT = DATA / 'cursovay'
OUT.mkdir(parents=True, exist_ok=True)

SEED = 42
VAL_LIMIT = 1000
TRAIN_CONTEXTS = 2000
TRAIN_BM25_K = 500
VALID_BM25_K = 8000
FINAL_BM25_K = 8000
FINAL_TOP_K = 50

SEARCH = [
    'search_query',
    'search_location_id',
    'search_is_delivery_search',
    'search_infm_params_text',
    'search_category',
]

TEXT = [
    'item_title_raw',
    'item_infm_params_text',
    'item_description_raw',
]

META = [
    'item_id',
    'item_title_raw',
    'item_infm_params_text',
    'item_category_id',
    'item_microcat_id',
    'item_location_id',
    'item_price',
    'item_rating',
    'item_rating_reviews_count',
    'item_is_phone_hidden',
    'item_is_message_forbidden',
]

required_files = [
    DATA / 'train.parquet',
    DATA / 'benchmark_queries.parquet',
    DATA / 'benchmark_items.parquet',
]
missing_files = [str(p) for p in required_files if not p.exists()]
if missing_files:
    raise FileNotFoundError(f'Не найдены входные файлы: {missing_files}')

train = pd.read_parquet(
    DATA / 'train.parquet',
    columns=SEARCH + ['item_id'] + TEXT,
)
benchmark_items_text = pd.read_parquet(
    DATA / 'benchmark_items.parquet',
    columns=['item_id'] + TEXT,
)
queries = pd.read_parquet(DATA / 'benchmark_queries.parquet')

print('train:', len(train))
print('benchmark_items:', len(benchmark_items_text))
print('benchmark_queries:', len(queries))


## 2. Train/validation split

Разделение выполняется по нормализованному search_query, поэтому одинаковый текст запроса не попадает одновременно в train и validation. Для локальной оценки корпус состоит из benchmark_items и уникальных объявлений из train, чтобы релевантные объявления validation гарантированно присутствовали в поисковом корпусе.


In [ ]:
# Разделение по тексту запроса.
query_groups = train['search_query'].fillna('').str.lower().str.strip()
unique_groups = np.sort(query_groups.unique())
rng = np.random.default_rng(SEED)
held_out = rng.choice(
    unique_groups,
    size=max(1, len(unique_groups) // 5),
    replace=False,
)
val_mask = query_groups.isin(held_out)

fit_small = train.loc[~val_mask, SEARCH + ['item_id']].copy()
val_small = train.loc[val_mask, SEARCH + ['item_id']].copy()

# Один контекст запроса может иметь несколько релевантных item_id.
validation = (
    val_small
    .groupby(SEARCH, dropna=False, observed=True, sort=False)['item_id']
    .agg(lambda x: sorted(set(x)))
    .reset_index(name='relevant')
)
validation = validation.sample(
    n=min(VAL_LIMIT, len(validation)),
    random_state=SEED,
).reset_index(drop=True)
validation['qid'] = np.arange(len(validation), dtype=np.int32)

train_contexts = (
    fit_small
    .groupby(SEARCH, dropna=False, observed=True, sort=False)['item_id']
    .agg(lambda x: sorted(set(x)))
    .reset_index(name='relevant')
)
train_contexts = train_contexts.sample(
    n=min(TRAIN_CONTEXTS, len(train_contexts)),
    random_state=SEED,
).reset_index(drop=True)
train_contexts['qid'] = np.arange(len(train_contexts), dtype=np.int32)

# Проверочный retrieval-корпус. benchmark_items имеет приоритет при одинаковом item_id.
corpus = (
    pd.concat(
        [benchmark_items_text, train[['item_id'] + TEXT]],
        ignore_index=True,
    )
    .drop_duplicates('item_id')
    .sort_values('item_id')
    .reset_index(drop=True)
)

train_contexts.to_parquet(OUT / 'train_contexts.parquet', index=False)
validation.to_parquet(OUT / 'val_contexts.parquet', index=False)

print('Train-контекстов:', len(train_contexts))
print('Validation-контекстов:', len(validation))
print('Документов в validation-корпусе:', len(corpus))

# Больше эти временные объекты не нужны.
del query_groups, unique_groups, held_out, val_mask, fit_small, val_small
gc.collect()


## 3. BM25 и кандидаты для обучения/валидации

BM25 строится по объединённому тексту `title + params + description`. Для обучения CatBoost используется top-500 BM25, но в память не загружаются все отрицательные пары: сохраняются positives, 20 hard negatives и до 30 случайных negatives. Для validation сразу сохраняем top-8000 — это тот размер пула, который используется в финальном решении. Он получен экспериментальным путем.


In [ ]:
stemmer = Stemmer.Stemmer('russian')

def tokenize(text):
    text = '' if pd.isna(text) else str(text)
    words = re.findall(r'[а-яa-z0-9]+', text.lower().replace('ё', 'е'))
    return stemmer.stemWords(words)


def build_bm25(frame):
    tokens = [
        tokenize(' '.join('' if pd.isna(x) else str(x) for x in row))
        for row in frame[TEXT].itertuples(index=False, name=None)
    ]
    model = bm25s.BM25(k1=1.5, b=0.75, method='lucene')
    model.index(tokens, show_progress=False)
    return model, frame['item_id'].to_numpy(dtype='U16')


def stable_topk_positions(scores, k):
    """Top-k по score DESC с тем же разрешением ничьих, что stable argsort."""
    scores = np.asarray(scores)
    n = len(scores)
    if k >= n:
        return np.argsort(-scores, kind='stable')

    threshold = np.partition(scores, n - k)[n - k]
    greater = np.flatnonzero(scores > threshold)
    need = k - len(greater)
    equal = np.flatnonzero(scores == threshold)[:need]
    positions = np.concatenate([greater, equal])
    order = np.lexsort((positions, -scores[positions]))
    return positions[order]


candidate_schema = pa.schema([
    ('qid', pa.int32()),
    ('item_id', pa.string()),
    ('bm25', pa.float32()),
    ('bm25_rank', pa.int32()),
    ('label', pa.int8()),
])


def export_candidates(contexts, filename, k, training=False):
    rng = np.random.default_rng(SEED)
    path = OUT / filename
    tmp = path.with_suffix('.tmp')

    recall50 = []
    recall_pool = []
    skipped = 0
    rows_written = 0

    with pq.ParquetWriter(tmp, candidate_schema, compression='zstd') as writer:
        for n, row in enumerate(contexts.itertuples(index=False), start=1):
            q_tokens = list(dict.fromkeys(tokenize(row.search_query)))
            scores = (
                bm25_model.get_scores(q_tokens)
                if q_tokens
                else np.zeros(len(bm25_ids), dtype=np.float32)
            )

            positions = stable_topk_positions(scores, k)
            ids = bm25_ids[positions]
            relevant = set(row.relevant)
            labels = np.fromiter(
                (item_id in relevant for item_id in ids),
                dtype=np.int8,
                count=len(ids),
            )

            recall50.append(labels[:FINAL_TOP_K].sum() / len(relevant))
            recall_pool.append(labels.sum() / len(relevant))

            keep = np.arange(len(ids))
            if training:
                positive = np.flatnonzero(labels)
                negative = np.flatnonzero(labels == 0)

                # Если positive не попал в BM25 top-500, этот query не даёт обучающего сигнала.
                if len(positive) == 0 or len(negative) == 0:
                    skipped += 1
                    continue

                hard_negative = negative[:20]
                other_negative = negative[20:]
                random_negative = (
                    rng.choice(other_negative, size=min(30, len(other_negative)), replace=False)
                    if len(other_negative)
                    else np.array([], dtype=np.int64)
                )
                keep = np.sort(
                    np.concatenate([positive, hard_negative, random_negative])
                )

            table = pa.Table.from_pydict(
                {
                    'qid': np.full(len(keep), row.qid, dtype=np.int32),
                    'item_id': ids[keep].tolist(),
                    'bm25': scores[positions[keep]].astype(np.float32),
                    'bm25_rank': (keep + 1).astype(np.int32),
                    'label': labels[keep],
                },
                schema=candidate_schema,
            )
            # Один row group соответствует одному запросу.
            writer.write_table(table)
            rows_written += len(keep)

            if n % 100 == 0:
                print(f'{filename}: {n}/{len(contexts)}')

    tmp.replace(path)
    print(
        f'{filename}: строк={rows_written}, skipped={skipped}, '
        f'Recall@50={np.mean(recall50):.6f}, Recall@{k}={np.mean(recall_pool):.6f}'
    )
    return float(np.mean(recall50)), float(np.mean(recall_pool))


print('Строим BM25 для validation-корпуса...')
bm25_model, bm25_ids = build_bm25(corpus)
print('BM25 готов:', len(bm25_ids), 'документов')

# Train-кандидаты для обучения деревьев.
export_candidates(
    train_contexts,
    'train_candidates.parquet',
    TRAIN_BM25_K,
    training=True,
)

# Validation-кандидаты сразу в финальном размере пула.
val_bm25_r50, val_bm25_pool = export_candidates(
    validation,
    'val_candidates_8000.parquet',
    VALID_BM25_K,
    training=False,
)

# BM25 и большие текстовые таблицы дальше не нужны.
del bm25_model, bm25_ids, corpus, train, benchmark_items_text
gc.collect()


## 4. Признаки и CatBoostRanker

К BM25-score и rank добавляются категории, география, цена, рейтинг, признаки пропусков и простые лексические пересечения запроса/фильтров с title и params. label, qid и item_id в модель не передаются.


In [ ]:
def load_candidate_metadata(candidate_files):
    """Загружает metadata только для item_id, реально попавших в кандидаты."""
    needed = set()
    for candidate_file in candidate_files:
        for batch in pq.ParquetFile(candidate_file).iter_batches(
            batch_size=32768,
            columns=['item_id'],
        ):
            needed.update(batch.column(0).to_pylist())

    parts = []
    remaining = needed.copy()

    # benchmark_items имеет приоритет при одинаковом item_id.
    for source in [DATA / 'benchmark_items.parquet', DATA / 'train.parquet']:
        parquet = pq.ParquetFile(source)
        for batch in parquet.iter_batches(batch_size=8192, columns=META):
            frame = batch.to_pandas()
            mask = frame['item_id'].isin(remaining)
            if mask.any():
                take = frame.loc[mask].drop_duplicates('item_id').copy()
                parts.append(take)
                remaining.difference_update(take['item_id'])
            if not remaining:
                break
        if not remaining:
            break

    if remaining:
        raise ValueError(f'Не найдены metadata для {len(remaining)} кандидатов')

    result = (
        pd.concat(parts, ignore_index=True)
        .drop_duplicates('item_id')
        .set_index('item_id')
    )
    print('Metadata загружены:', len(result))
    return result


metadata = load_candidate_metadata([
    OUT / 'train_candidates.parquet',
    OUT / 'val_candidates_8000.parquet',
])


def terms(value):
    text = '' if pd.isna(value) else str(value)
    words = re.findall(r'[а-яa-z0-9]+', text.lower().replace('ё', 'е'))
    return set(stemmer.stemWords(words))


def category(value):
    if pd.isna(value):
        return '__MISSING__'
    if (
        isinstance(value, (int, float, np.integer, np.floating))
        and np.isfinite(value)
        and float(value).is_integer()
    ):
        return str(int(value))
    return str(value)


CATS = [
    'search_category',
    'item_category_id',
    'item_microcat_id',
    'search_location_id',
    'item_location_id',
    'search_delivery',
    'category_pair',
    'location_pair',
]


def make_features(pairs, context):
    meta = metadata.loc[pairs['item_id']].reset_index(drop=True)
    x = pd.DataFrame(index=np.arange(len(pairs)))

    # Retrieval-признаки.
    x['bm25'] = pairs['bm25'].to_numpy(dtype=np.float32)
    x['bm25_rank'] = pairs['bm25_rank'].to_numpy(dtype=np.float32)
    query_terms = terms(context['search_query'])
    x['bm25_per_query_term'] = x['bm25'] / max(1, len(query_terms))

    # Категории и география.
    for name in ['item_category_id', 'item_microcat_id', 'item_location_id']:
        x[name] = meta[name].map(category)

    x['search_category'] = category(context['search_category'])
    x['search_location_id'] = category(context['search_location_id'])
    x['search_delivery'] = category(context['search_is_delivery_search'])
    x['category_pair'] = x['search_category'] + '|' + x['item_category_id']
    x['location_pair'] = x['search_location_id'] + '|' + x['item_location_id']

    known_location = (
        x['search_location_id'].ne('__MISSING__')
        & x['item_location_id'].ne('__MISSING__')
    )
    x['same_location'] = (
        known_location & x['search_location_id'].eq(x['item_location_id'])
    ).astype('float32')

    for name in [
        'search_category', 'item_category_id', 'item_microcat_id',
        'search_location_id', 'item_location_id', 'search_delivery',
    ]:
        x[name + '_missing'] = x[name].eq('__MISSING__').astype('float32')

    # Числовые признаки объявления.
    for name in [
        'item_price',
        'item_rating',
        'item_rating_reviews_count',
        'item_is_phone_hidden',
        'item_is_message_forbidden',
    ]:
        values = (
            pd.to_numeric(meta[name], errors='coerce')
            .replace([np.inf, -np.inf], np.nan)
        )
        x[name] = values.astype('float32')
        x[name + '_missing'] = values.isna().astype('float32')

    # Лексическое соответствие query/filter полям объявления.
    filter_terms = terms(context['search_infm_params_text'])
    params = [terms(t) for t in meta['item_infm_params_text']]
    titles = [terms(t) for t in meta['item_title_raw']]

    x['filter_empty'] = np.float32(not filter_terms)
    x['params_empty'] = np.array([not t for t in params], dtype=np.float32)
    x['title_empty'] = np.array([not t for t in titles], dtype=np.float32)

    for name, left, right in [
        ('filter_params', filter_terms, params),
        ('filter_title', filter_terms, titles),
        ('query_params', query_terms, params),
        ('query_title', query_terms, titles),
    ]:
        x[name] = np.array(
            [len(left & t) / max(1, len(left)) for t in right],
            dtype=np.float32,
        )

    return x


In [ ]:
# Обучение CatBoostRanker на сохранённых train-кандидатах.
train_contexts_idx = train_contexts.set_index('qid')
val_contexts_idx = validation.set_index('qid')

pf = pq.ParquetFile(OUT / 'train_candidates.parquet')
blocks, labels, group_ids = [], [], []

for i in range(pf.num_row_groups):
    pairs = pf.read_row_group(i).to_pandas()
    qid = int(pairs['qid'].iloc[0])
    blocks.append(make_features(pairs, train_contexts_idx.loc[qid]))
    labels.append(pairs['label'].to_numpy())
    group_ids.append(np.full(len(pairs), qid, dtype=np.int32))

X = pd.concat(blocks, ignore_index=True)
y = np.concatenate(labels)
groups = np.concatenate(group_ids)

print('Пар для обучения:', len(X))
print('Групп:', len(np.unique(groups)))

pool = Pool(X, label=y, group_id=groups, cat_features=CATS)
ranker = CatBoostRanker(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    loss_function='YetiRank',
    task_type='CPU',
    thread_count=2,
    random_seed=SEED,
    max_ctr_complexity=1,
    used_ram_limit='2gb',
    verbose=False,
    allow_writing_files=False,
)
ranker.fit(pool)
ranker.save_model(str(OUT / 'ranker.cbm'))
(OUT / 'feature_names.json').write_text(
    json.dumps(X.columns.tolist(), ensure_ascii=False),
    encoding='utf-8',
)

feature_names = X.columns.tolist()
del X, y, groups, pool, blocks, labels, group_ids, pairs
gc.collect()

print('Модель сохранена:', OUT / 'ranker.cbm')


## 5. Локальная проверка итогового каскада

На каждом validation-запросе CatBoost оценивает все 8000 BM25-кандидатов и оставляет 50. Эта проверка соответствует выбранному финальному размеру candidate pool.


In [ ]:
pf = pq.ParquetFile(OUT / 'val_candidates_8000.parquet')
metrics = []

for i in range(pf.num_row_groups):
    pairs = pf.read_row_group(i).to_pandas()
    qid = int(pairs['qid'].iloc[0])
    context = val_contexts_idx.loc[qid]
    truth = set(context['relevant'])

    features = make_features(pairs, context)
    if features.columns.tolist() != feature_names:
        raise ValueError('Порядок признаков отличается от обучающей выборки')

    rank_scores = ranker.predict(features)
    top50_pos = np.argsort(-rank_scores, kind='stable')[:FINAL_TOP_K]
    top50_ids = pairs.iloc[top50_pos]['item_id'].tolist()

    metrics.append({
        'qid': qid,
        'bm25_recall50': len(truth.intersection(pairs.iloc[:FINAL_TOP_K]['item_id'])) / len(truth),
        'bm25_recall8000': len(truth.intersection(pairs['item_id'])) / len(truth),
        'reranked_recall50': len(truth.intersection(top50_ids)) / len(truth),
    })

    if (i + 1) % 100 == 0:
        print(f'Validation: {i + 1}/{pf.num_row_groups}')

validation_result = pd.DataFrame(metrics)
validation_result.to_csv(OUT / 'validation_reranked.csv', index=False)

print(f"BM25 Recall@50:             {validation_result['bm25_recall50'].mean():.6f}")
print(f"BM25 Recall@8000:           {validation_result['bm25_recall8000'].mean():.6f}")
print(f"BM25 top-8000 -> CatBoost:  {validation_result['reranked_recall50'].mean():.6f}")


## 6. Финальный benchmark и `answer.csv`

Для отправки BM25 перестраивается только на `benchmark_items.parquet`, потому что платформа разрешает возвращать только item_id из этого корпуса. CatBoost уже обучен; для каждого benchmark-запроса он выбирает 50 объявлений из BM25 top-8000.


In [ ]:
# Освобождаем validation metadata перед финальным корпусом.
del metadata
gc.collect()

final_item_columns = list(dict.fromkeys(['item_id'] + TEXT + META))
benchmark_items = pd.read_parquet(
    DATA / 'benchmark_items.parquet',
    columns=final_item_columns,
)
benchmark_items = benchmark_items.sort_values('item_id').reset_index(drop=True)

print('Строим финальный BM25 на benchmark_items...')
final_bm25, final_item_ids = build_bm25(benchmark_items)

# Для CatBoost description уже не нужен.
metadata = (
    benchmark_items[META]
    .drop_duplicates('item_id')
    .set_index('item_id')
)
del benchmark_items
gc.collect()

predictions = []
query_ids = []

for n, (_, context) in enumerate(queries.iterrows(), start=1):
    q_tokens = list(dict.fromkeys(tokenize(context['search_query'])))
    bm25_scores = (
        final_bm25.get_scores(q_tokens)
        if q_tokens
        else np.zeros(len(final_item_ids), dtype=np.float32)
    )

    positions = stable_topk_positions(bm25_scores, FINAL_BM25_K)
    pairs = pd.DataFrame({
        'item_id': final_item_ids[positions],
        'bm25': bm25_scores[positions].astype(np.float32),
        'bm25_rank': np.arange(1, len(positions) + 1, dtype=np.int32),
    })

    features = make_features(pairs, context)
    rank_scores = ranker.predict(features)
    best = np.argsort(-rank_scores, kind='stable')[:FINAL_TOP_K]

    predictions.append(pairs.iloc[best]['item_id'].tolist())
    query_ids.append(context['query_id'])

    if n % 100 == 0:
        gc.collect()
        print(f'Benchmark: {n}/{len(queries)}')

answer = pd.DataFrame({
    'query_id': query_ids,
    'answer': [' '.join(ids) for ids in predictions],
})

answer.to_csv(DATA / 'answer.csv', index=False, encoding='utf-8')
print('Сохранено:', DATA / 'answer.csv')
